# Build Texas ZCTA ACS Target Profiles

This notebook builds one ACS target-profile row per Texas ZCTA. The output is used later to control demographic sampling from PUMS donor profiles: ACS targets preserve local ZIP/ZCTA distributions, while PUMS later preserves realistic combinations of age, education, income, household structure, occupation, and homeownership.

This notebook does not create synthetic customers and does not pull PUMS data.

## 1. Imports and Configuration

Paths are built with `pathlib`. The notebook is restartable and uses cached raw ACS files unless `FORCE_REFRESH` is set to `True`.

In [ ]:
from pathlib import Path
import json
import math
import os
import time

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path("Synthetic MySQL Tables") / "Customers"
if not PROJECT_ROOT.exists() and Path.cwd().name == "Customers":
    PROJECT_ROOT = Path(".")

RAW_DIR = PROJECT_ROOT / "raw" / "census"
PROCESSED_DIR = PROJECT_ROOT / "processed" / "census"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ACS_YEAR = 2024
ACS_DATASET = "acs/acs5"
STATE_FIPS = "48"
BASE_API_URL = f"https://api.census.gov/data/{ACS_YEAR}/{ACS_DATASET}"
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
FORCE_REFRESH = False

OUTPUT_MAIN = PROCESSED_DIR / "tx_zcta_acs_targets_2024.csv"
OUTPUT_AUDIT = PROCESSED_DIR / "tx_zcta_acs_targets_2024_audit.csv"
OUTPUT_MISSING_ZCTAS = PROCESSED_DIR / "missing_zctas_from_acs_2024.csv"

ZIP_COLUMN_CANDIDATES = ["zcta", "zip_code", "zip", "assigned_zip", "customer_zip_code", "ZIP", "ZCTA"]
TEXAS_ZCTA_PREFIXES = ("75", "76", "77", "78", "79", "88")

TARGET_ZCTA_FILE_CANDIDATES = [
    PROJECT_ROOT / "processed" / "census" / "tx_zcta_to_puma_crosswalk_2020.csv",
    PROJECT_ROOT / "tx_zcta_to_puma_crosswalk_2020.csv",
    PROJECT_ROOT / "texas_target_weights.csv",
    PROJECT_ROOT / "postal_customers.csv",
    PROJECT_ROOT / "data" / "processed" / "census" / "tx_zcta_to_puma_crosswalk_2020.csv",
    Path("Synthetic_Customers_and_Orders") / "postal_customers.csv",
    Path("Establish_source_codes_and_target_weights") / "texas_target_weights.csv",
]

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Raw ACS cache: {RAW_DIR.resolve()}")
print(f"Processed output: {PROCESSED_DIR.resolve()}")
print(f"Census API key available: {bool(CENSUS_API_KEY)}")

## 2. Define ACS Variables

The variables below come from the 2024 ACS 5-year Detailed Tables API. They are grouped by source table so each raw cache file remains easy to audit.

In [ ]:
ACS_VARIABLE_GROUPS = {
    "B19001": {
        "description": "Household income in the past 12 months",
        "variables": [f"B19001_{i:03d}E" for i in range(1, 18)],
    },
    "B25003": {
        "description": "Tenure",
        "variables": ["B25003_001E", "B25003_002E", "B25003_003E"],
    },
    "B11001": {
        "description": "Household type",
        "variables": ["B11001_001E", "B11001_002E", "B11001_003E"],
    },
    "B15003": {
        "description": "Educational attainment for population 25 years and over",
        "variables": [f"B15003_{i:03d}E" for i in range(1, 26)],
    },
    "B01001": {
        "description": "Sex by age",
        "variables": [
            *[f"B01001_{i:03d}E" for i in range(7, 26)],
            *[f"B01001_{i:03d}E" for i in range(31, 50)],
        ],
    },
}

ALL_ACS_VARIABLES = sorted({variable for group in ACS_VARIABLE_GROUPS.values() for variable in group["variables"]})
print(f"ACS variable groups: {list(ACS_VARIABLE_GROUPS)}")
print(f"Unique ACS estimate variables: {len(ALL_ACS_VARIABLES):,}")

## 3. Pull ACS Data

Each table is cached as a raw CSV. If the Census API request list ever gets too long, the fetch helper splits variables into chunks and merges them back on geography.

In [ ]:
def chunked(values: list[str], size: int) -> list[list[str]]:
    return [values[index:index + size] for index in range(0, len(values), size)]


def fetch_acs_json(params: dict, retries: int = 4, backoff_seconds: float = 1.5) -> list[list[str]]:
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_API_URL, params=params, timeout=60)
            response.raise_for_status()
            if "missing_key.html" in response.url:
                raise RuntimeError("The Census API requested a valid key for this endpoint. Set CENSUS_API_KEY and rerun, or use cached raw ACS files.")
            if not response.text.lstrip().startswith("["):
                excerpt = response.text[:300].replace("\n", " ")
                raise RuntimeError(f"ACS API returned a non-JSON response from {response.url}: {excerpt}")
            return response.json()
        except Exception as error:
            last_error = error
            wait_seconds = backoff_seconds * attempt
            print(f"ACS request failed on attempt {attempt}/{retries}: {error}. Retrying in {wait_seconds:.1f}s")
            time.sleep(wait_seconds)
    raise RuntimeError(f"ACS request failed after {retries} attempts") from last_error


def acs_json_to_frame(payload: list[list[str]]) -> pd.DataFrame:
    if not payload or len(payload) < 2:
        raise ValueError("ACS API returned no data rows")
    return pd.DataFrame(payload[1:], columns=payload[0])


def fetch_acs_table(year: int, dataset: str, variables: list[str], geography: str, api_key: str | None = None) -> pd.DataFrame:
    del year, dataset
    geog_column = geography.split(":", 1)[0]
    frames = []

    for variable_chunk in chunked(variables, 45):
        params = {
            "get": ",".join(["NAME", *variable_chunk]),
            "for": geography,
        }
        if api_key:
            params["key"] = api_key

        payload = fetch_acs_json(params)
        frames.append(acs_json_to_frame(payload))

    merged = frames[0]
    for frame in frames[1:]:
        merged = merged.merge(frame, on=["NAME", geog_column], how="outer")
    return merged


def load_or_fetch_group(group_name: str, variables: list[str]) -> pd.DataFrame:
    cache_path = RAW_DIR / f"acs5_{ACS_YEAR}_{group_name}_zcta_raw.csv"
    if cache_path.exists() and not FORCE_REFRESH:
        print(f"Loading cached {group_name}: {cache_path}")
        return pd.read_csv(cache_path, dtype="string", low_memory=False)

    print(f"Fetching {group_name} from ACS API")
    frame = fetch_acs_table(
        year=ACS_YEAR,
        dataset=ACS_DATASET,
        variables=variables,
        geography="zip code tabulation area:*",
        api_key=CENSUS_API_KEY,
    )
    frame.to_csv(cache_path, index=False)
    print(f"Cached {group_name}: {cache_path} ({len(frame):,} rows)")
    return frame


raw_acs_tables = {
    group_name: load_or_fetch_group(group_name, group_config["variables"])
    for group_name, group_config in ACS_VARIABLE_GROUPS.items()
}

## 4. Normalize and Merge ACS Tables

The ACS geography field is normalized to `zcta`, and all estimate columns are converted to numeric values before target calculations.

In [ ]:
def normalize_acs_table(frame: pd.DataFrame) -> pd.DataFrame:
    normalized = frame.copy()
    geog_columns = [column for column in normalized.columns if column.lower() == "zip code tabulation area"]
    if not geog_columns:
        raise ValueError("Expected ACS geography column 'zip code tabulation area'")

    geog_column = geog_columns[0]
    normalized["zcta"] = normalized[geog_column].astype("string").str.strip().str.zfill(5)
    normalized = normalized.drop(columns=[geog_column])
    return normalized


normalized_tables = {group_name: normalize_acs_table(frame) for group_name, frame in raw_acs_tables.items()}

acs_merged = None
for group_name, frame in normalized_tables.items():
    if acs_merged is None:
        acs_merged = frame.copy()
    else:
        acs_merged = acs_merged.merge(frame, on=["zcta", "NAME"], how="outer")
    print(f"Merged {group_name}: {len(acs_merged):,} rows")

acs_merged = acs_merged.rename(columns={"NAME": "zcta_name"})
for column in ALL_ACS_VARIABLES:
    if column in acs_merged.columns:
        acs_merged[column] = pd.to_numeric(acs_merged[column], errors="coerce")

print(f"Merged ACS ZCTA rows before Texas filtering: {len(acs_merged):,}")

## 5. Filter to Texas Target ZCTAs

When available, a local crosswalk or customer file defines the target ZCTA list. If no local list exists, the notebook falls back to Texas-like ZCTA prefixes (`75`, `76`, `77`, `78`, `79`, `88`). Prefix filtering is only a fallback; the crosswalk or target ZIP list is preferred.

In [ ]:
def infer_zip_column(columns: pd.Index) -> str | None:
    exact_lookup = {column.lower(): column for column in columns}
    for candidate in ZIP_COLUMN_CANDIDATES:
        if candidate.lower() in exact_lookup:
            return exact_lookup[candidate.lower()]
    return None


def load_target_zctas() -> tuple[set[str] | None, Path | None, str | None]:
    for candidate_path in TARGET_ZCTA_FILE_CANDIDATES:
        if not candidate_path.exists():
            continue

        target_frame = pd.read_csv(candidate_path, dtype="string", low_memory=False)
        zip_column = "zcta" if "zcta" in target_frame.columns else infer_zip_column(target_frame.columns)
        if zip_column is None:
            print(f"Found {candidate_path}, but no ZIP/ZCTA column was recognized.")
            continue

        target_zctas = set(target_frame[zip_column].dropna().astype("string").str.strip().str.zfill(5))
        print(f"Using target ZCTA list from: {candidate_path}")
        print(f"Inferred ZIP/ZCTA column: {zip_column}")
        print(f"Target ZCTA count: {len(target_zctas):,}")
        return target_zctas, candidate_path, zip_column

    print("No local target ZCTA file found. Falling back to Texas-like ZCTA prefixes.")
    return None, None, None


target_zctas, target_zcta_path, target_zcta_column = load_target_zctas()

if target_zctas is None:
    filtered_acs = acs_merged.loc[acs_merged["zcta"].str.startswith(TEXAS_ZCTA_PREFIXES, na=False)].copy()
    used_target_file = False
else:
    filtered_acs = acs_merged.loc[acs_merged["zcta"].isin(target_zctas)].copy()
    used_target_file = True

filtered_acs["state_fips"] = STATE_FIPS

print(f"ACS ZCTAs before filtering: {acs_merged['zcta'].nunique():,}")
print(f"ACS rows after filtering: {len(filtered_acs):,}")
print(f"Filtered ZCTAs: {filtered_acs['zcta'].nunique():,}")

## 6. Calculate Target Proportions

The target categories map the detailed ACS variables into the simplified customer fields used by the postal demo schema.

In [ ]:
def sum_columns(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    return frame[columns].sum(axis=1, min_count=1)


def safe_divide(numerator, denominator) -> pd.Series:
    numerator_series = pd.Series(numerator, index=denominator.index if hasattr(denominator, "index") else None, dtype="float64")
    denominator_series = pd.Series(denominator, index=numerator_series.index, dtype="float64")
    valid = denominator_series.notna() & denominator_series.gt(0)
    result = pd.Series(np.nan, index=numerator_series.index, dtype="float64")
    result.loc[valid] = numerator_series.loc[valid] / denominator_series.loc[valid]
    return result


def clip_pct_columns(frame: pd.DataFrame, pct_columns: list[str]) -> pd.DataFrame:
    clipped = frame.copy()
    clipped_any = pd.Series(False, index=clipped.index)
    for column in pct_columns:
        clipped_flag = clipped[column].notna() & ~clipped[column].between(0, 1)
        clipped[f"{column}_was_clipped"] = clipped_flag.astype(int)
        clipped_any = clipped_any | clipped_flag
        clipped[column] = clipped[column].clip(0, 1)
    clipped["any_pct_was_clipped"] = clipped_any.astype(int)
    return clipped


targets = filtered_acs.copy()

targets["income_total"] = targets["B19001_001E"]
targets["income_low_count"] = sum_columns(targets, [f"B19001_{i:03d}E" for i in range(2, 11)])
targets["income_medium_count"] = sum_columns(targets, ["B19001_011E", "B19001_012E", "B19001_013E"])
targets["income_high_count"] = sum_columns(targets, ["B19001_014E", "B19001_015E"])
targets["income_very_high_count"] = sum_columns(targets, ["B19001_016E", "B19001_017E"])

targets["tenure_total"] = targets["B25003_001E"]
targets["owner_count"] = targets["B25003_002E"]
targets["renter_count"] = targets["B25003_003E"]

targets["household_type_total"] = targets["B11001_001E"]
targets["married_household_count"] = targets["B11001_003E"]

targets["education_total"] = targets["B15003_001E"]
targets["partial_high_school_count"] = sum_columns(targets, [f"B15003_{i:03d}E" for i in range(2, 17)])
targets["high_school_count"] = sum_columns(targets, ["B15003_017E", "B15003_018E"])
targets["partial_college_count"] = sum_columns(targets, ["B15003_019E", "B15003_020E", "B15003_021E"])
targets["bachelors_count"] = targets["B15003_022E"]
targets["graduate_degree_count"] = sum_columns(targets, ["B15003_023E", "B15003_024E", "B15003_025E"])

age_18_24_columns = ["B01001_007E", "B01001_008E", "B01001_009E", "B01001_010E", "B01001_031E", "B01001_032E", "B01001_033E", "B01001_034E"]
age_25_34_columns = ["B01001_011E", "B01001_012E", "B01001_035E", "B01001_036E"]
age_35_44_columns = ["B01001_013E", "B01001_014E", "B01001_037E", "B01001_038E"]
age_45_64_columns = ["B01001_015E", "B01001_016E", "B01001_017E", "B01001_018E", "B01001_019E", "B01001_039E", "B01001_040E", "B01001_041E", "B01001_042E", "B01001_043E"]
age_65_plus_columns = ["B01001_020E", "B01001_021E", "B01001_022E", "B01001_023E", "B01001_024E", "B01001_025E", "B01001_044E", "B01001_045E", "B01001_046E", "B01001_047E", "B01001_048E", "B01001_049E"]
age_18_plus_columns = age_18_24_columns + age_25_34_columns + age_35_44_columns + age_45_64_columns + age_65_plus_columns

targets["age_18_plus_count"] = sum_columns(targets, age_18_plus_columns)
targets["age_18_24_count"] = sum_columns(targets, age_18_24_columns)
targets["age_25_34_count"] = sum_columns(targets, age_25_34_columns)
targets["age_35_44_count"] = sum_columns(targets, age_35_44_columns)
targets["age_45_64_count"] = sum_columns(targets, age_45_64_columns)
targets["age_65_plus_count"] = sum_columns(targets, age_65_plus_columns)

targets["income_low_pct"] = safe_divide(targets["income_low_count"], targets["income_total"])
targets["income_medium_pct"] = safe_divide(targets["income_medium_count"], targets["income_total"])
targets["income_high_pct"] = safe_divide(targets["income_high_count"], targets["income_total"])
targets["income_very_high_pct"] = safe_divide(targets["income_very_high_count"], targets["income_total"])
targets["owner_pct"] = safe_divide(targets["owner_count"], targets["tenure_total"])
targets["renter_pct"] = safe_divide(targets["renter_count"], targets["tenure_total"])
targets["married_household_pct"] = safe_divide(targets["married_household_count"], targets["household_type_total"])
targets["partial_high_school_pct"] = safe_divide(targets["partial_high_school_count"], targets["education_total"])
targets["high_school_pct"] = safe_divide(targets["high_school_count"], targets["education_total"])
targets["partial_college_pct"] = safe_divide(targets["partial_college_count"], targets["education_total"])
targets["bachelors_pct"] = safe_divide(targets["bachelors_count"], targets["education_total"])
targets["graduate_degree_pct"] = safe_divide(targets["graduate_degree_count"], targets["education_total"])
targets["age_18_24_pct"] = safe_divide(targets["age_18_24_count"], targets["age_18_plus_count"])
targets["age_25_34_pct"] = safe_divide(targets["age_25_34_count"], targets["age_18_plus_count"])
targets["age_35_44_pct"] = safe_divide(targets["age_35_44_count"], targets["age_18_plus_count"])
targets["age_45_64_pct"] = safe_divide(targets["age_45_64_count"], targets["age_18_plus_count"])
targets["age_65_plus_pct"] = safe_divide(targets["age_65_plus_count"], targets["age_18_plus_count"])

targets["total_households"] = targets["income_total"]
targets["source_year"] = ACS_YEAR
targets["source_dataset"] = ACS_DATASET

PCT_COLUMNS = [column for column in targets.columns if column.endswith("_pct")]
targets = clip_pct_columns(targets, PCT_COLUMNS)

print(f"Calculated target profiles: {len(targets):,} rows")

## 7. Final Tables

The main CSV keeps the simple join-ready target schema. The audit CSV keeps counts, denominators, clipping flags, and raw ACS estimates.

In [ ]:
FINAL_COLUMNS = [
    "zcta",
    "zcta_name",
    "state_fips",
    "total_households",
    "income_low_pct",
    "income_medium_pct",
    "income_high_pct",
    "income_very_high_pct",
    "owner_pct",
    "renter_pct",
    "married_household_pct",
    "partial_high_school_pct",
    "high_school_pct",
    "partial_college_pct",
    "bachelors_pct",
    "graduate_degree_pct",
    "age_18_24_pct",
    "age_25_34_pct",
    "age_35_44_pct",
    "age_45_64_pct",
    "age_65_plus_pct",
    "source_year",
    "source_dataset",
]

COUNT_AND_DENOMINATOR_COLUMNS = [
    "income_total",
    "income_low_count",
    "income_medium_count",
    "income_high_count",
    "income_very_high_count",
    "tenure_total",
    "owner_count",
    "renter_count",
    "household_type_total",
    "married_household_count",
    "education_total",
    "partial_high_school_count",
    "high_school_count",
    "partial_college_count",
    "bachelors_count",
    "graduate_degree_count",
    "age_18_plus_count",
    "age_18_24_count",
    "age_25_34_count",
    "age_35_44_count",
    "age_45_64_count",
    "age_65_plus_count",
]

CLIP_FLAG_COLUMNS = [column for column in targets.columns if column.endswith("_was_clipped") or column == "any_pct_was_clipped"]
RAW_ACS_COLUMNS = [column for column in ALL_ACS_VARIABLES if column in targets.columns]
AUDIT_COLUMNS = FINAL_COLUMNS + COUNT_AND_DENOMINATOR_COLUMNS + CLIP_FLAG_COLUMNS + RAW_ACS_COLUMNS

final_targets = targets[FINAL_COLUMNS].copy().sort_values("zcta").reset_index(drop=True)
audit_targets = targets[AUDIT_COLUMNS].copy().sort_values("zcta").reset_index(drop=True)

for column in PCT_COLUMNS:
    if column in final_targets.columns:
        final_targets[column] = final_targets[column].round(6)

for column in COUNT_AND_DENOMINATOR_COLUMNS + RAW_ACS_COLUMNS:
    if column in audit_targets.columns:
        audit_targets[column] = audit_targets[column].round().astype("Int64")

print(f"Final target rows: {len(final_targets):,}")
print(f"Audit columns: {len(audit_targets.columns):,}")

## 8. Validation Checks

The notebook fails on structural issues and warns for sparse ACS denominators. Small denominator problems can happen for low-population ZCTAs, so those are reported without stopping the pipeline.

In [ ]:
def report_denominator_issues(frame: pd.DataFrame) -> pd.DataFrame:
    denominator_columns = ["income_total", "tenure_total", "household_type_total", "education_total", "age_18_plus_count"]
    issue_mask = pd.Series(False, index=frame.index)
    for column in denominator_columns:
        issue_mask = issue_mask | frame[column].isna() | frame[column].le(0)
    issues = frame.loc[issue_mask, ["zcta", "zcta_name", *denominator_columns]].copy()
    print(f"Rows with missing or zero denominators: {len(issues):,}")
    if not issues.empty:
        display(issues.head(25))
    return issues


def validate_group_sum(frame: pd.DataFrame, columns: list[str], label: str, tolerance: float = 0.01) -> None:
    group_sum = frame[columns].sum(axis=1, min_count=len(columns))
    bad = frame.loc[group_sum.notna() & ~np.isclose(group_sum, 1.0, atol=tolerance), ["zcta", "zcta_name", *columns]].copy()
    print(f"{label} group-sum warnings: {len(bad):,}")
    if not bad.empty:
        bad[f"{label}_sum"] = group_sum.loc[bad.index]
        display(bad.head(25))


def validate_targets(final_frame: pd.DataFrame, audit_frame: pd.DataFrame) -> None:
    assert list(final_frame.columns) == FINAL_COLUMNS, "Final columns are not in the requested order"
    assert final_frame["zcta"].astype("string").str.len().eq(5).all(), "zcta must always be 5 characters"
    assert not final_frame["zcta"].duplicated().any(), "zcta must not have duplicates"
    assert final_frame["state_fips"].eq(STATE_FIPS).all(), "state_fips must always be 48"
    assert pd.api.types.is_numeric_dtype(final_frame["total_households"]), "total_households must be numeric"
    assert final_frame["source_year"].eq(ACS_YEAR).all(), "source_year must be 2024"

    pct_columns = [column for column in final_frame.columns if column.endswith("_pct")]
    pct_ranges_ok = final_frame[pct_columns].apply(lambda series: series.dropna().between(0, 1).all()).all()
    assert pct_ranges_ok, "All pct columns must be between 0 and 1, ignoring nulls"

    report_denominator_issues(audit_frame)
    null_pct_rows = final_frame.loc[final_frame[pct_columns].isna().any(axis=1), ["zcta", "zcta_name", *pct_columns]]
    print(f"Rows with any null final pct columns: {len(null_pct_rows):,}")
    if not null_pct_rows.empty:
        display(null_pct_rows.head(25))

    validate_group_sum(final_frame, ["income_low_pct", "income_medium_pct", "income_high_pct", "income_very_high_pct"], "income")
    validate_group_sum(final_frame, ["owner_pct", "renter_pct"], "tenure")
    validate_group_sum(final_frame, ["partial_high_school_pct", "high_school_pct", "partial_college_pct", "bachelors_pct", "graduate_degree_pct"], "education")
    validate_group_sum(final_frame, ["age_18_24_pct", "age_25_34_pct", "age_35_44_pct", "age_45_64_pct", "age_65_plus_pct"], "age")

    print("Structural validation complete.")


validate_targets(final_targets, audit_targets)

if used_target_file:
    matched_zctas = set(final_targets["zcta"])
    missing_zctas = sorted(target_zctas - matched_zctas)
    print(f"Target ZCTAs: {len(target_zctas):,}")
    print(f"Matched in ACS: {len(matched_zctas):,}")
    print(f"Missing from ACS: {len(missing_zctas):,}")
    missing_frame = pd.DataFrame({"zcta": missing_zctas})
    missing_frame.to_csv(OUTPUT_MISSING_ZCTAS, index=False)
    if missing_zctas:
        display(missing_frame.head(50))
        print(f"Saved missing target ZCTAs: {OUTPUT_MISSING_ZCTAS}")

## 9. Save Outputs

In [ ]:
def save_outputs(final_frame: pd.DataFrame, audit_frame: pd.DataFrame) -> None:
    final_frame.to_csv(OUTPUT_MAIN, index=False)
    audit_frame.to_csv(OUTPUT_AUDIT, index=False)
    print(f"Saved main ACS targets: {OUTPUT_MAIN}")
    print(f"Saved audit ACS targets: {OUTPUT_AUDIT}")


save_outputs(final_targets, audit_targets)

## 10. Examples and Summary Statistics

In [ ]:
display(final_targets.head(10))

In [ ]:
example_zctas = ["77494", "77002", "75201", "78701", "79936"]
for example_zcta in example_zctas:
    rows = final_targets.loc[final_targets["zcta"].eq(example_zcta)]
    if rows.empty:
        print(f"ZCTA {example_zcta}: not present in final targets.")
    else:
        print(f"ZCTA {example_zcta}")
        display(rows)

In [ ]:
summary_stats = pd.DataFrame(
    {
        "metric": ["number of ZCTAs", "mean total_households", "median total_households"],
        "value": [
            final_targets["zcta"].nunique(),
            round(final_targets["total_households"].mean(), 2),
            round(final_targets["total_households"].median(), 2),
        ],
    }
)
display(summary_stats)

print("Top 10 ZCTAs by total_households")
display(final_targets.sort_values("total_households", ascending=False).head(10))

print("Top 10 ZCTAs by income_very_high_pct")
display(final_targets.sort_values("income_very_high_pct", ascending=False).head(10))

print("Top 10 ZCTAs by owner_pct")
display(final_targets.sort_values("owner_pct", ascending=False).head(10))

print("Top 10 ZCTAs by graduate_degree_pct")
display(final_targets.sort_values("graduate_degree_pct", ascending=False).head(10))

## Final Interpretation

Each row represents one Texas ZCTA. The proportion columns are demographic targets for later synthetic customer generation. For example, if ZCTA 77494 has 1,000 synthetic postal customers and `owner_pct = 0.78`, then the later demographic sampler should aim for roughly 780 homeowners among customers assigned to that ZIP.

These ACS targets should be used to control ZIP-level distributions, while ACS PUMS donor records will later preserve realistic combinations of age, education, income, household structure, occupation, and homeownership.